# Exploratory Data Analysis (EDA)

Analyze AgentBench game data: statistics, patterns, behavioral features.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# Connect to database
db_path = Path('data/game_data.db')
conn = sqlite3.connect(db_path)

# Load games and rounds
games_df = pd.read_sql_query('SELECT * FROM games', conn)
rounds_df = pd.read_sql_query('SELECT * FROM rounds', conn)

print(f'Loaded {len(games_df)} games and {len(rounds_df)} rounds')
print(f'\nGames DataFrame:\n{games_df.head()}')
print(f'\nRounds DataFrame:\n{rounds_df.head()}')

## Game-Level Statistics

In [ ]:
# Summary statistics
print('='*60)
print('GAME SUMMARY STATISTICS')
print('='*60)

print(f'\nTotal Games: {len(games_df)}')
print(f'Total Rounds: {len(rounds_df)}')
print(f'Avg Game Length: {games_df["n_turns"].mean():.1f} turns')
print(f'Median Game Length: {games_df["n_turns"].median():.1f} turns')

# Win/loss/tie distribution
games_df['outcome'] = np.where(games_df['agent_score'] > games_df['opponent_score'], 'Win',
                               np.where(games_df['agent_score'] < games_df['opponent_score'], 'Loss', 'Tie'))

outcomes = games_df['outcome'].value_counts()
print(f'\nOutcome Distribution:')
print(outcomes)
print(f'\nWin Rate: {(outcomes.get("Win", 0) / len(games_df) * 100):.1f}%')

## Agent Performance by Opponent

In [ ]:
# Win rate by opponent
print('\nWin Rate vs Each Opponent:')
for opp in games_df['opponent_name'].unique():
    opp_games = games_df[games_df['opponent_name'] == opp]
    wins = (opp_games['agent_score'] > opp_games['opponent_score']).sum()
    wr = wins / len(opp_games) * 100
    print(f'  {opp:15s}: {wr:5.1f}% ({wins}/{len(opp_games)})')

# Visualization
opponent_stats = []
for opp in games_df['opponent_name'].unique():
    opp_games = games_df[games_df['opponent_name'] == opp]
    wins = (opp_games['agent_score'] > opp_games['opponent_score']).sum()
    opponent_stats.append({'opponent': opp, 'win_rate': wins/len(opp_games)})

opponent_df = pd.DataFrame(opponent_stats)
fig = px.bar(opponent_df, x='opponent', y='win_rate', title='Win Rate vs Each Opponent')
fig.show()

## Round-Level Analysis

In [ ]:
# Move distribution
MOVE_NAMES = ['ROCK', 'PAPER', 'SCISSORS', 'LIZARD', 'POWER', 'RECHARGE']

agent_moves = rounds_df['agent_move'].value_counts().sort_index()
opponent_moves = rounds_df['opponent_move'].value_counts().sort_index()

print('\nAgent Move Distribution:')
for move_id, count in agent_moves.items():
    pct = count / len(rounds_df) * 100
    print(f'  {MOVE_NAMES[move_id]:10s}: {count:6d} ({pct:5.1f}%)')

print('\nOpponent Move Distribution:')
for move_id, count in opponent_moves.items():
    pct = count / len(rounds_df) * 100
    print(f'  {MOVE_NAMES[move_id]:10s}: {count:6d} ({pct:5.1f}%)')

In [ ]:
# Visualize move distributions
agent_dist_df = pd.DataFrame({
    'Move': [MOVE_NAMES[i] for i in agent_moves.index],
    'Count': agent_moves.values,
    'Type': 'Agent'
})

opponent_dist_df = pd.DataFrame({
    'Move': [MOVE_NAMES[i] for i in opponent_moves.index],
    'Count': opponent_moves.values,
    'Type': 'Opponent'
})

dist_df = pd.concat([agent_dist_df, opponent_dist_df])
fig = px.bar(dist_df, x='Move', y='Count', color='Type', barmode='group',
             title='Move Distribution: Agent vs Opponent')
fig.show()

## Energy Usage Analysis

In [ ]:
# Energy patterns
print('\nAgent Energy After Round (Average):')
print(rounds_df['agent_energy_after'].describe())

print('\nOpponent Energy After Round (Average):')
print(rounds_df['opponent_energy_after'].describe())

# Energy trends by turn
energy_by_turn = rounds_df.groupby('turn')[['agent_energy_after', 'opponent_energy_after']].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=energy_by_turn.index, y=energy_by_turn['agent_energy_after'],
                          mode='lines', name='Agent Energy'))
fig.add_trace(go.Scatter(x=energy_by_turn.index, y=energy_by_turn['opponent_energy_after'],
                          mode='lines', name='Opponent Energy'))
fig.update_layout(title='Average Energy by Turn', xaxis_title='Turn', yaxis_title='Energy')
fig.show()

## Outcome Analysis

In [ ]:
# Outcome counts
outcomes = rounds_df['outcome'].value_counts()
print('\nRound Outcomes:')
for outcome, count in outcomes.items():
    pct = count / len(rounds_df) * 100
    print(f'  {outcome:5s}: {count:6d} ({pct:5.1f}%)')

# Visualization
fig = px.pie(values=outcomes.values, names=outcomes.index, title='Round Outcome Distribution')
fig.show()

## Behavioral Feature Extraction

In [ ]:
# Move transitions (what move follows what move)
print('\nMove Transition Analysis:')
transitions = {}

for game_id in rounds_df['game_id'].unique():
    game_rounds = rounds_df[rounds_df['game_id'] == game_id].sort_values('turn')
    moves = game_rounds['agent_move'].values
    
    for i in range(len(moves) - 1):
        from_move = int(moves[i])
        to_move = int(moves[i+1])
        key = (from_move, to_move)
        transitions[key] = transitions.get(key, 0) + 1

# Top transitions
print('\nTop 10 Move Transitions:')
for (from_m, to_m), count in sorted(transitions.items(), key=lambda x: -x[1])[:10]:
    print(f'  {MOVE_NAMES[from_m]:10s} -> {MOVE_NAMES[to_m]:10s}: {count:4d}')

## Data Quality Checks

In [ ]:
# Validation
print('DATA QUALITY CHECKS')
print('='*60)

# Check for missing values
print('\nMissing values in games_df:')
print(games_df.isnull().sum())

print('\nMissing values in rounds_df:')
print(rounds_df.isnull().sum())

# Check move validity
invalid_agent_moves = rounds_df[~rounds_df['agent_move'].isin(range(6))]
invalid_opponent_moves = rounds_df[~rounds_df['opponent_move'].isin(range(6))]

print(f'\nInvalid agent moves: {len(invalid_agent_moves)}')
print(f'Invalid opponent moves: {len(invalid_opponent_moves)}')

# Check energy validity
invalid_energy = rounds_df[
    (rounds_df['agent_energy_after'] < 0) | (rounds_df['agent_energy_after'] > 5) |
    (rounds_df['opponent_energy_after'] < 0) | (rounds_df['opponent_energy_after'] > 5)
]
print(f'Invalid energy values: {len(invalid_energy)}')

print('\n✅ Data quality check complete!')

## Summary Insights

In [ ]:
print('\nKEY INSIGHTS')
print('='*60)

# Agent strategy
power_rate = (rounds_df['agent_move'] == 4).sum() / len(rounds_df) * 100
recharge_rate = (rounds_df['agent_move'] == 5).sum() / len(rounds_df) * 100

print(f'\n1. Agent Strategy:')
print(f'   - POWER usage: {power_rate:.1f}%')
print(f'   - RECHARGE usage: {recharge_rate:.1f}%')

# Game length
print(f'\n2. Game Duration:')
print(f'   - Average: {games_df["n_turns"].mean():.0f} turns')
print(f'   - Max: {games_df["n_turns"].max()} turns')
print(f'   - Min: {games_df["n_turns"].min()} turns')

# Performance
print(f'\n3. Performance:')
avg_win_rate = (games_df['agent_score'] > games_df['opponent_score']).sum() / len(games_df) * 100
print(f'   - Overall win rate: {avg_win_rate:.1f}%')

conn.close()
print('\n✅ EDA complete!')